In [29]:
import requests
import os
from pathlib import Path
import psycopg2
from skimpy import skim
import pandas as pd
from collections import defaultdict


In [30]:
conn=psycopg2.connect(user="postgres",host="localhost",port="5432",password="jaune2000",database="sandbox_db")
cur=conn.cursor()

In [31]:
def json_structure(obj, indent=0):
    
    prefix = "  " * indent
    if isinstance(obj, dict):
        for k, v in obj.items():
            print(f"{prefix}{k}: {type(v).__name__}")
            if isinstance(v, (dict, list)):
                json_structure(v, indent + 1)
    elif isinstance(obj, list) and obj:
        print(f"{prefix}[list of {len(obj)}]")
        json_structure(obj[0], indent + 1)  # inspect first item


In [32]:
def create_tables(cur):
    cur.execute("DROP TABLE IF EXISTS station_info")
    query="""CREATE TABLE station_info(
    station_id BIGINT  PRIMARY KEY,
    station_code VARCHAR(5) CHECK (length(station_code)=5),
    name VARCHAR(255),
    lat REAL,
    lon REAL,
    capacity INTEGER,
    rental_method VARCHAR(255)
    )"""
    cur.execute(query)

    cur.execute("DROP TABLE IF EXISTS station_status")
    query="""CREATE TABLE station_status(
    station_id BIGINT PRIMARY KEY,
    num_docks_available INTEGER,
    num_bikes_available INTEGER,
    is_installed BOOLEAN,
    is_returning BOOLEAN,
    is_renting BOOLEAN,
    last_reported BIGINT ,
    station_code VARCHAR CHECK(length(station_code)=5),
    num_bikes_mechanical INTEGER,
    num_bikes_ebike INTEGER
    )"""
    cur.execute(query)


In [ ]:
cur.execute("SELECT i.station_id,name,num_bikes_available FROM station_info i INNER JOIN station_status s ON i.station_id=s.station_id ORDER BY num_bikes_available")
print(cur.fetchall())


[(7304, 'Bassano -  Iéna', 0), (559980654, 'George V - Christophe Colomb', 0), (20689981670, 'Roossevelt - Impasse du Marché', 0), (478732841, 'Sebastopol - Rambuteau', 0), (10773448319, 'Clef des Champs - Albert Dhalenne', 0), (74805344, 'Pierre Charron - Champs-Elysées', 0), (76352106, 'Saint-Augustin', 0), (567884788, 'Vincent Auriol - Louise Weiss', 0), (210408693, 'Vaugirard - Monsieur le Prince', 0), (27363256, 'Thérèse - Opéra', 0), (266422568, 'Place Edmond Michelet', 0), (82584215, 'Place Lafayette', 0), (2705356759, 'Jean Jaurès - Paul Lafargue', 0), (17486828694, 'Charlot - Stade Gabriel Voisin', 0), (75345531, 'Turbigo - Française', 0), (128932790, 'Arsène Houssaye - Champs-Elysées', 0), (17174675707, 'Saint Pères - Saint Germain', 0), (1062807847, 'BNF - Bibliothèque Nationale de France', 0), (6245, 'Ventadour - Opéra', 0), (128843085, 'Place du Lieutenant Henri Karcher', 0), (128986726, 'Boétie - Ponthieu', 0), (276989966, 'Oratoire - Rivoli', 0), (37521, 'Petites Ecuries

In [59]:
cur.execute("SELECT i.station_id,name,num_docks_available FROM station_info i INNER JOIN station_status s ON i.station_id=s.station_id ORDER BY num_docks_available")
print(cur.fetchall())

[(63498079, 'Marché Secrétan', 0), (102338493, 'Larmeroux - Baudelaire', 0), (101743895, 'Joseph Liouville - Croix Nivert', 0), (213694482, 'Botzaris - Parc des Buttes Chaumont', 0), (54000616, 'Pyrénées - Avron', 0), (21372728303, 'Place Daydé - Ile Seguin', 0), (9931940126, 'Diane Arbus', 0), (128932790, 'Arsène Houssaye - Champs-Elysées', 0), (235513935, 'Carducci - Place Hannah Arendt', 0), (102324803, 'Pyrénées - Emmery', 0), (20689981670, 'Roossevelt - Impasse du Marché', 0), (101751231, 'Saint-Amand - Labrouste', 0), (102323152, "Manin - Carrières d'Amérique", 0), (66491385, 'Fontenay - Libération', 0), (213987795, 'Wilson - Landy', 0), (13718041, 'Square Simon Bolivar', 0), (20682017769, 'Pierre-Simon Girard  - Paris', 0), (19495443023, 'Paul Bodin - Clichy', 0), (82332239, 'Ecoles - Carmes', 0), (129096696, 'Léon - Doudeauville', 0), (18976782828, 'Welovegreen', 0), (213682736, 'Morillons - Dantzig', 0), (54000619, 'Porte de Bagnolet', 0), (1057360420, 'Place Aimé Césaire', 0)

In [73]:
cur.execute("SELECT SUM(num_docks_available)* 1.0/ SUM(num_bikes_available) AS mean_disponibility FROM station_status")
print(cur.fetchall())

[(Decimal('0.62053526342366604949'),)]


In [ ]:
cur.execute("SELECT SUM(num_docks_available) FROM station_status")
print(cur.fetchall())  

[(0,)]


In [34]:
def get_station_info_data():
    url_info_station="https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_information.json"
    response_json=requests.get(url_info_station).json()
    json_structure(response_json)
    #print(response_json)
    #skim(pd.DataFrame(response_json["data"]))
    return response_json

In [35]:
def get_station_status_data():
    url_station_statut="https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_status.json"
    response_json=requests.get(  url_station_statut).json()
    json_structure(response_json)
    #skim(pd.DataFrame(response_json["data"]))
    return response_json


In [36]:
def store_data():
    
    pass

In [37]:
create_tables(cur)

In [38]:
station_info_json=get_station_info_data()
station_status_json=get_station_status_data()
station_info_json=station_info_json["data"]["stations"]
for dict_row in station_info_json:
    dict_row.pop("station_opening_hours",None)



lastUpdatedOther: int
ttl: int
data: dict
  stations: list
    [list of 1517]
      station_id: int
      stationCode: str
      name: str
      lat: float
      lon: float
      capacity: int
      station_opening_hours: NoneType
lastUpdatedOther: int
ttl: int
data: dict
  stations: list
    [list of 1517]
      station_id: int
      num_bikes_available: int
      numBikesAvailable: int
      num_bikes_available_types: list
        [list of 2]
          mechanical: int
      num_docks_available: int
      numDocksAvailable: int
      is_installed: int
      is_returning: int
      is_renting: int
      last_reported: int
      stationCode: str
      station_opening_hours: NoneType


In [39]:
for dict_row in station_info_json:
    if dict_row["stationCode"][0]=="0":
        print(dict_row["stationCode"])

In [40]:
df_station_info = pd.json_normalize(station_info_json)
df_station_info.head(1)

,station_id,stationCode,name,lat,lon,capacity,rental_methods
0,213688169,16107,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,NaN


In [41]:
df_station_info = pd.json_normalize(station_info_json)
df_station_info

,station_id,stationCode,name,lat,lon,capacity,rental_methods
0,213688169,16107,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,NaN
1,19179944124,40001,Hôpital Mondor,48.798922,2.453745,28,NaN
2,18795078746,32304,Charcot - Benfleet,48.878370,2.440524,28,NaN
3,36255,9020,Toudouze - Clauzel,48.879296,2.337360,21,[CREDITCARD]
4,251039991,14111,Cassini - Denfert-Rochereau,48.837526,2.336035,25,[CREDITCARD]
...,...,...,...,...,...,...,...
1512,523900881,19018,Cité de la Musique,48.888797,2.392581,60,[CREDITCARD]
1513,34742973,15056,Place Balard,48.836396,2.278419,22,[CREDITCARD]
1514,101013666,12105,Bercy - Villot,48.841795,2.376785,33,[CREDITCARD]
1515,315022587,8004,Malesherbes - Place de la Madeleine,48.870406,2.323244,67,[CREDITCARD]


In [42]:
df_station_status = pd.json_normalize(station_status_json["data"]["stations"])
df_station_status=df_station_status.drop(columns=["station_opening_hours","numDocksAvailable","numBikesAvailable"])
df_station_status

,station_id,num_bikes_available,num_bikes_available_types,num_docks_available,is_installed,is_returning,is_renting,last_reported,stationCode
0,213688169,2,"[{'mechanical': 1}, {'ebike': 1}]",33,1,1,1,1780998645,16107
1,19179944124,19,"[{'mechanical': 10}, {'ebike': 9}]",9,1,1,1,1780998687,40001
2,18795078746,1,"[{'mechanical': 0}, {'ebike': 1}]",27,1,1,1,1780998201,32304
3,36255,3,"[{'mechanical': 1}, {'ebike': 2}]",17,1,1,1,1780998638,9020
4,251039991,3,"[{'mechanical': 0}, {'ebike': 3}]",21,1,1,1,1780998551,14111
...,...,...,...,...,...,...,...,...,...
1512,523900881,20,"[{'mechanical': 8}, {'ebike': 12}]",39,1,1,1,1780998658,19018
1513,34742973,31,"[{'mechanical': 28}, {'ebike': 3}]",0,1,1,1,1780998726,15056
1514,101013666,29,"[{'mechanical': 21}, {'ebike': 8}]",4,1,1,1,1780998741,12105
1515,315022587,76,"[{'mechanical': 45}, {'ebike': 31}]",3,1,1,1,1780998692,8004


In [43]:
def pad_to_five(df): 
    new_station_code=[]
    for val in df["stationCode"].sort_values(ascending=True):
        if len(val)!=5:
            new_station_code.append((5-len(val))*"0"+val)
        else:
            new_station_code.append(val)
    df["stationCode"]=new_station_code
    return df

In [44]:
df_station_info=pad_to_five(df_station_info)
df_station_status=pad_to_five(df_station_status)

In [45]:
num_mechanicals_list=[]
num_ebikes_list=[]
for value in df_station_status["num_bikes_available_types"]:

    num_mechanicals_list.append(value[0]["mechanical"])
    num_ebikes_list.append(value[1]["ebike"])

df_station_status["num_mech_bikes"]=num_mechanicals_list
df_station_status["num_ebikes"]=num_ebikes_list
df_station_status=df_station_status.drop(columns=["num_bikes_available_types"])
df_station_status.head(1)

,station_id,num_bikes_available,num_docks_available,is_installed,is_returning,is_renting,last_reported,stationCode,num_mech_bikes,num_ebikes
0,213688169,2,33,1,1,1,1780998645,10003,1,1


In [46]:
station_status_insert=df_station_status.to_numpy().tolist()
station_status_insert[0:2]

[[213688169, 2, 33, 1, 1, 1, 1780998645, '10003', 1, 1],
 [19179944124, 19, 9, 1, 1, 1, 1780998687, '10004', 10, 9]]

In [47]:
cur.executemany("INSERT INTO station_status VALUES(%s,%s,%s,%s::boolean,%s::boolean,%s::boolean,%s,%s,%s,%s)", station_status_insert)

In [48]:
df_station_info.head(2)

,station_id,stationCode,name,lat,lon,capacity,rental_methods
0,213688169,10003,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,NaN
1,19179944124,10004,Hôpital Mondor,48.798922,2.453745,28,NaN


In [49]:
station_info_to_insert=df_station_info.to_numpy().tolist()
station_info_to_insert[0:2]

[[213688169,
  '10003',
  'Benjamin Godard - Victor Hugo',
  48.865983,
  2.275725,
  35,
  nan],
 [19179944124,
  '10004',
  'Hôpital Mondor',
  48.798922410229,
  2.4537451531298,
  28,
  nan]]

In [50]:
cur.executemany("INSERT INTO station_info VALUES(%s,%s,%s,%s,%s,%s,%s)", station_info_to_insert)

In [51]:
#cur.execute("ROLLBACK")

In [67]:
cur.execute("SELECT * FROM station_info LIMIT 3")
print(cur.fetchall())

cur.execute("SELECT * FROM station_status LIMIT 3")
print(cur.fetchall())

[(213688169, '10003', 'Benjamin Godard - Victor Hugo', 48.865982, 2.275725, 35, 'NaN'), (19179944124, '10004', 'Hôpital Mondor', 48.798923, 2.4537451, 28, 'NaN'), (18795078746, '10005', 'Charcot - Benfleet', 48.87837, 2.4405239, 28, 'NaN')]
[(213688169, 2, 33, True, True, True, 1780998645, '10003', 1, 1), (19179944124, 19, 9, True, True, True, 1780998687, '10004', 10, 9), (18795078746, 1, 27, True, True, True, 1780998201, '10005', 0, 1)]


In [53]:
"""
1. Les stations les plus vides et les plus pleines (avec leur nom, donc avec une jointure).
2. Le taux de disponibilité moyen     (vélos rapportés à la capacité).
3. Les stations de grande capacité. """

'\n1. Les stations les plus vides et les plus pleines (avec leur nom, donc avec une jointure).\n2. Le taux de disponibilité moyen     (vélos rapportés à la capacité).\n3. Les stations de grande capacité. '